# 08 — Anchor-age feature + training-time masking augmentation (P1)

Per the Opus-model review's P1 recommendation: add `months_since_anchor` (how
stale each masked cell's backward-fill anchor is) so the model can distinguish a
1-month-ahead forecast from a 4-month-ahead one off the same frozen anchor value.

This feature is only *learnable* if training data actually shows it varying.
Train.csv is otherwise always fully observed, so `months_since_anchor` would be a
constant 0 for every fit row without deliberately augmenting the fit set with
simulated masking too - not just the validation fold, as `mask_aware_horizon_
matched_split` (notebook 07) already does.

`evaluate.mask_augmented_horizon_matched_split` and `features.compute_anchor_age`
are already implemented and unit-tested in `src/` - this notebook is their
validation, per this project's graduation rule. Checked over 5 independent
masking realisations (seeds), not just one, since a single realisation could be
a lucky/unlucky draw.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config, data, evaluate, features, model
from src.train import get_feature_cols

train, test, sample_submission = data.load_raw_data()
masked_month_fraction, masked_row_fraction = evaluate.measure_masking_pattern(test)
target_horizons = evaluate.compute_test_horizons(train, test)
print(f"masked_month_fraction={masked_month_fraction:.3f} masked_row_fraction={masked_row_fraction:.3f}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



masked_month_fraction=0.667 masked_row_fraction=0.998


## Reference: current production proxy (no fit augmentation)

The number `src/train.py` currently reports.

In [2]:
def score(fit, val, cols):
    X_fit = features.select_base_features(fit, cols)
    y_fit = fit[config.TARGET_COL].to_numpy()
    X_val = features.select_base_features(val, cols)
    y_val = val[config.TARGET_COL].to_numpy()
    m = model.make_baseline_model()
    m.fit(X_fit, y_fit)
    y_pred = model.predict(m, X_val)
    return evaluate.compute_metrics(y_val, y_pred)


fit_ref, val_ref = evaluate.mask_aware_horizon_matched_split(
    train, target_horizons, masked_month_fraction, masked_row_fraction
)
feature_cols = get_feature_cols(fit_ref)
ref_metrics = score(fit_ref, val_ref, feature_cols)
print("Reference (mask-aware, fit NOT augmented):", ref_metrics)

C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

Reference (mask-aware, fit NOT augmented): {'rmse': 0.7469201070251745, 'mae': 0.544200311000805, 'r2': 0.24856174071178838}


## Augmented fit, with and without `months_since_anchor` - 5 independent seeds

Each seed draws an independent masking realisation for both the fit augmentation
and the validation masking. Same model, same feature set otherwise - only
(a) whether fit is augmented and (b) whether `months_since_anchor` is included
differ.

In [3]:
results = []
for seed in range(5):
    fit_aug, val_aug = evaluate.mask_augmented_horizon_matched_split(
        train, target_horizons, masked_month_fraction, masked_row_fraction,
        fit_seed=seed, val_seed=seed + 100,
    )
    no_age = score(fit_aug, val_aug, feature_cols)
    with_age = score(fit_aug, val_aug, feature_cols + ["months_since_anchor"])
    results.append({"seed": seed, **{f"no_age_{k}": v for k, v in no_age.items()},
                     **{f"with_age_{k}": v for k, v in with_age.items()}})

results_df = pd.DataFrame(results).set_index("seed")
results_df

,no_age_rmse,no_age_mae,no_age_r2,with_age_rmse,with_age_mae,with_age_r2
seed,,,,,,
0,0.731529,0.537143,0.279210,0.727911,0.533250,0.286324
1,0.729252,0.545341,0.283691,0.725103,0.537548,0.291819
2,0.730741,0.539622,0.280764,0.714880,0.524710,0.311647
3,0.702000,0.519954,0.336228,0.691546,0.508879,0.355850
4,0.707094,0.524176,0.326559,0.703326,0.519981,0.333718


In [4]:
summary = pd.DataFrame({
    "reference (no augmentation)": ref_metrics,
    "augmented, no age (mean over 5 seeds)": {
        "rmse": results_df["no_age_rmse"].mean(), "mae": results_df["no_age_mae"].mean(),
        "r2": results_df["no_age_r2"].mean(),
    },
    "augmented + anchor age (mean over 5 seeds)": {
        "rmse": results_df["with_age_rmse"].mean(), "mae": results_df["with_age_mae"].mean(),
        "r2": results_df["with_age_r2"].mean(),
    },
}).T
summary["rmse_improvement_vs_reference"] = summary.loc["reference (no augmentation)", "rmse"] - summary["rmse"]
summary

,rmse,mae,r2,rmse_improvement_vs_reference
reference (no augmentation),0.746920,0.544200,0.248562,0.000000
"augmented, no age (mean over 5 seeds)",0.720123,0.533247,0.301290,0.026797
augmented + anchor age (mean over 5 seeds),0.712553,0.524874,0.315871,0.034367


In [5]:
wins_no_age = (results_df["no_age_rmse"] < ref_metrics["rmse"]).sum()
wins_with_age_vs_no_age = (results_df["with_age_rmse"] < results_df["no_age_rmse"]).sum()
print(f"Fit augmentation beats the unaugmented reference on RMSE in {wins_no_age}/5 seeds")
print(f"Adding anchor age beats augmented-without-age on RMSE in {wins_with_age_vs_no_age}/5 seeds")

Fit augmentation beats the unaugmented reference on RMSE in 5/5 seeds
Adding anchor age beats augmented-without-age on RMSE in 5/5 seeds


## Gate decision

**WIN, on both changes, robust across all 5 seeds — graduating.**

1. **Training-time masking augmentation alone** (no anchor-age feature yet) beats
   the unaugmented reference on RMSE in 5/5 seeds, and on every metric component
   in every seed but one (a single MAE blip 0.2% worse than reference in one
   seed, corrected by adding the age feature in that same seed - see the raw
   per-seed table above). Mean RMSE improvement over the reference: see the
   summary table. This is the "train the way you'll be tested" fix the P0 gap
   analysis implied but didn't itself implement - the model now actually sees
   frozen-anchor examples during fitting, not just at real inference time.
2. **`months_since_anchor` on top of the augmented fit** beats augmented-without-
   age on RMSE in 5/5 seeds too, with no exceptions on any metric component in
   any seed - a much cleaner result than the trend feature's trade-offs.

**Why this is judged safer than the trend feature** (which also looked good
internally and then failed on the real leaderboard): trend read the recent
`TWS_t` *trajectory* and extrapolated it - exactly the mechanism P0 showed the
proxy misjudges. Anchor age and the augmentation methodology do the opposite:
they make the model's training distribution match its real deployment
distribution more closely, which is the same class of fix that P0 itself was.
Still, per the Opus review's Tier A/Tier B policy, this is being treated
cautiously - graduated into `src/`, but a real Zindi submission
(`outputs/submission_anchor_age.csv`) is generated and queued for upload before
fully trusting it as a repeatable win.

**Graduated:** `features.compute_anchor_age` and
`evaluate.mask_augmented_horizon_matched_split` (already implemented,
unit-tested). `src/train.py`'s final production fit now trains on
masking-augmented Train.csv with `months_since_anchor` included, matching this
validated methodology.